In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
df = pd.read_csv('Mall_Customers.csv')
print("Shape:", df.shape)
display(df.head(3))

# Features: Annual Income and Spending Score
X = df.iloc[:, [3, 4]].values
feature_names = [df.columns[3], df.columns[4]]
print("Features used:", feature_names)

In [ ]:
inertia = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km.fit(X)
    inertia.append(km.inertia_)

print("--- Inertia values (k = 1 to 10) ---")
for k, val in enumerate(inertia, start=1):
    print(f"  k = {k:2d} : {val:.2f}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), inertia, marker='o', color='purple')
plt.title('Elbow Method – Optimal k Selection')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(range(1, 11))
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
linked = linkage(X, method='ward')

plt.figure(figsize=(10, 5))
dendrogram(linked, truncate_mode='lastp', p=20,
           leaf_rotation=45, leaf_font_size=10, show_contracted=True)
plt.title('Dendrogram (Ward Linkage)')
plt.xlabel('Sample index / cluster size')
plt.ylabel('Euclidean distance')
plt.axhline(y=200, color='red', linestyle='--', label='Cut-off threshold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
k_opt = 5
kmeans = KMeans(n_clusters=k_opt, init='k-means++', random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X)
centroids = kmeans.cluster_centers_

print(f"Optimal k = {k_opt}  |  Final inertia: {kmeans.inertia_:.2f}")
print("\nCluster sizes:")
print(df['Cluster'].value_counts().sort_index())

In [ ]:
palette = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4']

plt.figure(figsize=(8, 6))
for c in range(k_opt):
    mask = df['Cluster'] == c
    plt.scatter(X[mask, 0], X[mask, 1],
                color=palette[c], label=f'Cluster {c}',
                s=60, edgecolors='k', linewidths=0.4)

plt.scatter(centroids[:, 0], centroids[:, 1],
            color='black', marker='X', s=200, zorder=5, label='Centroids')
plt.title('K-Means Clustering (k = 5)')
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
summary = df.groupby('Cluster')[num_cols].mean().round(1)
display(summary)